# ReBRAC Broad Validation v2 — Core Training Driver (Colab) [rev.3 notebook]

> 文档对应：[`docs/rebrac_broad_validation_v2_plan.md`](../docs/rebrac_broad_validation_v2_plan.md) (rev.3 + cmd fix, commits `819ed3e` / `79294b8` / `7a4bc2a`)
>
> **本 notebook rev.3 (2026-05-19)** — 与 v1 broad val 路径约定对齐：
> - eval 数字 → `results/offline/rebrac/broad_validation_v2/<cell>/seed_<n>/test_result.json`（不再落在 checkpoint dir）
> - 汇总 → `results/offline/rebrac/broad_validation_v2/summaries/{verdict_decision.json, p1_overview.csv}`
> - checkpoint dir 只放 train artifacts（`agent_final.pt` / `trainer_state.json` / `train_log.jsonl`），重训资产；分析时**只取 `results/` 即可**
> - **自动 migration**：检测到 rev.1/rev.2 时代写到 checkpoint dir 的 `test_result.json` 会自动 copy 到 `results/` 新位置，4 个已跑完的 JSON 不需要重 eval
>
> **本 notebook rev.2 (2026-05-18) 已修**：
> - `subprocess.run → os.system` 让训练/评估输出实时 stream（rev.1 全缓冲）
> - `or` 短路 fix：成功率 = 0.0 不再被当 falsy 丢值
> - v1-style systematic results 表 + `verdict_decision.json` + `p1_overview.csv` 留痕
>
> 目的：跑完 plan rev.3 §6.2 的 **N0 (reward bridge)** + **N2' (oracle-teacher critical-regime)** 共 4 runs，应用 §5.2 / §5.3 verdict gate 决定是否触发 conditional M1 BC sweep。

## 执行摘要（首轮 2-seed Colab pass）

| Cell | Dataset | Manifest | Regime | Seeds | 用途 |
|---|---|---|---|---|---|
| **N0** | `crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000` | `single_u10_cross_tgt15` | sub-critical (U=1.0/Re=150) | 42, 0 | reward bridge anchor (vs main-line `efficiency_v2` 0.902 ± 0.021) |
| **N2'** | `privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000` | `single_u15_cross_tgt15` | critical (U=1.5/Re=250) | 42, 0 | oracle-teacher partial-obs ceiling probe (vs online catastrophic 0.10) |

## 输出位置（rev.3 对齐 v1）

| 类型 | 路径 | Sync 回 git? |
|---|---|---|
| Train checkpoint | `checkpoints/offline/rebrac/broad_validation_v2/<cell>/seed_<n>/{agent_final.pt, trainer_state.json, train_log.jsonl}` | 否（大文件，gitignore） |
| Eval 数字 | `results/offline/rebrac/broad_validation_v2/<cell>/seed_<n>/test_result.json` | **是** |
| 汇总 | `results/offline/rebrac/broad_validation_v2/summaries/{verdict_decision.json, p1_overview.csv}` | **是** |

**Verdict gates** (pre-registered)：

| Cell | Gate | 触发动作 |
|---|---|---|
| N0 | mean ≥ 0.70 → proceed N2' / 0.50–0.70 → weak / < 0.50 → **暂停所有后续** | §5.2 |
| N2' | mean ≥ 0.40 strong positive / [0.15, 0.40] partial → trigger M1 / < 0.15 strong negative | §5.3 |
| M1 | conditional on N2' ∈ [0.15, 0.40]，β1 ∈ {0,1,2,4,8} × 3 seed = 15 runs ≈ 15h L4 | 本 notebook 不跑 |

**Important caveat (rev.3 §2.4)**：privileged dataset 含 priv-action，但 ReBRAC actor 仍只看 s0_obs，学到的是 `E[priv_action | s0_obs]`，**不是** priv_action 本身。因此 N2' 数字封顶是 actor-fundamental ceiling under s0，不会真的达到 70%。

## 0. 环境 sanity

In [ ]:
!lscpu | head -8
print()
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device 0      : {torch.cuda.get_device_name(0)}")
    print(f"cuDNN         : {torch.backends.cudnn.version()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

In [ ]:
!pwd && ls scripts/train_offline.py docs/rebrac_broad_validation_v2_plan.md
!ls offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz \
    offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz
!ls benchmarks/single_u10_cross_tgt15.json benchmarks/single_u15_cross_tgt15.json

## 1. Run matrix

4 runs = 2 cell × 2 seed。每个 run 都是同一个 canonical `train_offline.py` invocation，只换 `--offline-data` / `--manifest` / `--seed` / `--save-dir` / `--output-json`。

Skip-resume：
- **train 阶段** 看 `<ckpt_dir>/trainer_state.json` + `<ckpt_dir>/agent_final.pt` 都存在则跳过
- **eval 阶段** 看 `<result_dir>/test_result.json` 存在则跳过

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path('.').resolve()
CKPT_ROOT = REPO_ROOT / 'checkpoints' / 'offline' / 'rebrac' / 'broad_validation_v2'
RESULT_ROOT = REPO_ROOT / 'results' / 'offline' / 'rebrac' / 'broad_validation_v2'
SUMMARIES_DIR = RESULT_ROOT / 'summaries'
SUMMARIES_DIR.mkdir(parents=True, exist_ok=True)

CELLS = {
    'N0': {
        'dataset': 'offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz',
        'manifest': 'benchmarks/single_u10_cross_tgt15.json',
        'regime_note': 'sub-critical (U=1.0/Re=150)',
    },
    'N2p': {
        'dataset': 'offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz',
        'manifest': 'benchmarks/single_u15_cross_tgt15.json',
        'regime_note': 'critical (U=1.5/Re=250)',
    },
}

SEEDS = [42, 0]

RUNS = []
for cell_id, cfg in CELLS.items():
    for seed in SEEDS:
        ckpt_dir = CKPT_ROOT / cell_id / f'seed_{seed}'
        result_dir = RESULT_ROOT / cell_id / f'seed_{seed}'
        RUNS.append({
            'cell_id': cell_id,
            'seed': seed,
            'dataset': cfg['dataset'],
            'manifest': cfg['manifest'],
            'ckpt_dir': str(ckpt_dir),
            'result_dir': str(result_dir),
            'regime_note': cfg['regime_note'],
        })

print(f'{"#":>2} {"cell":<5} {"seed":>5}  ckpt_dir / result_dir')
print('-' * 110)
for i, r in enumerate(RUNS, 1):
    print(f'{i:>2} {r["cell_id"]:<5} {r["seed"]:>5}  ckpt:   {r["ckpt_dir"]}')
    print(f'   {"":<5} {"":>5}  result: {r["result_dir"]}')

## 1.5 一次性 migration — rev.1/rev.2 时代写到 checkpoint dir 的 test_result.json → results/ 新位置

如果 Drive 上已经有 `checkpoints/.../seed_*/test_result.json`（rev.1 在 2026-05-18 跑完写在那里），本 cell 自动 copy 到新的 `results/.../seed_*/test_result.json`，省一次 100ep eval。**对全新跑 = no-op**。

In [ ]:
import shutil
import time

def _drive_retry(fn, *args, retries=5, delay=2, label='', **kwargs):
    """Retry on transient Drive FUSE errors (Errno 103 / 5 / 107 etc.)."""
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except OSError as e:
            if attempt == retries:
                raise
            print(f'  [retry {attempt}/{retries}] {label}: {type(e).__name__} errno={e.errno}; sleep {delay}s')
            time.sleep(delay)
            delay = min(delay * 2, 30)

for r in RUNS:
    old_test_json = Path(r['ckpt_dir']) / 'test_result.json'
    new_test_json = Path(r['result_dir']) / 'test_result.json'
    if _drive_retry(new_test_json.exists, label=f'stat {new_test_json}'):
        print(f'[skip] already in new location: {new_test_json}')
        continue
    if not _drive_retry(old_test_json.exists, label=f'stat {old_test_json}'):
        print(f'[noop] no legacy file at {old_test_json}')
        continue
    _drive_retry(new_test_json.parent.mkdir, parents=True, exist_ok=True,
                 label=f'mkdir {new_test_json.parent}')
    _drive_retry(shutil.copy2, old_test_json, new_test_json,
                 label=f'copy {old_test_json.name}')
    print(f'[migrate] {old_test_json} → {new_test_json}')

## 2. Training (4 runs, ~2–3h L4 total)

**rev.2 改 `subprocess.run → os.system`**：训练 epoch loss 实时 stream 到 cell（rev.1 全缓冲，cell 卡 6-8 min 一片空白）。

Canonical command (plan rev.3 §6.2, post-fix `7a4bc2a`):

- `--algo rebrac` (script 是 `scripts.train_offline` 不是 `scripts.train_offline_rebrac`)
- `--actor-penalty-coef 4.0 --critic-penalty-coef 2.0`
- `--sampling-mode shuffle_no_replacement --num-epochs 64`
- `--manifest`（不是 `--eval-manifest`）
- `--critic-layernorm --no-actor-layernorm`
- `--eval-every 0 --skip-final-eval`（评估在 §3 separately run）

In [ ]:
import shlex
import time

for i, r in enumerate(RUNS, 1):
    ckpt_dir = Path(r['ckpt_dir'])
    trainer_state = ckpt_dir / 'trainer_state.json'
    agent_final = ckpt_dir / 'agent_final.pt'

    print(f'\n========== [{i}/{len(RUNS)}] {r["cell_id"]} seed={r["seed"]} ({r["regime_note"]}) ==========')

    if trainer_state.exists() and agent_final.exists():
        print(f'[skip] already complete: {ckpt_dir}')
        continue

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    cmd_parts = [
        'python', '-u', '-m', 'scripts.train_offline',
        '--algo', 'rebrac',
        '--offline-data', r['dataset'],
        '--manifest', r['manifest'],
        '--probe-layout', 's0',
        '--history-length', '4',
        '--task-geometry', 'cross_stream',
        '--target-speed', '1.5',
        '--objective', 'arrival_v2',
        '--sampling-mode', 'shuffle_no_replacement',
        '--num-epochs', '64',
        '--batch-size', '256',
        '--hidden-dim', '256',
        '--num-hidden-layers', '3',
        '--actor-lr', '3e-4',
        '--critic-lr', '3e-4',
        '--gamma', '0.99',
        '--tau', '0.005',
        '--actor-penalty-coef', '4.0',
        '--critic-penalty-coef', '2.0',
        '--policy-noise', '0.2',
        '--noise-clip', '0.5',
        '--policy-freq', '2',
        '--grad-clip-norm', '10.0',
        '--normalizer-eps', '1e-3',
        '--critic-layernorm',
        '--no-actor-layernorm',
        '--eval-every', '0',
        '--skip-final-eval',
        '--log-every', '1000',
        '--seed', str(r['seed']),
        '--device', 'cuda',
        '--save-dir', str(ckpt_dir),
    ]
    cmd_str = ' '.join(shlex.quote(p) for p in cmd_parts)
    print(cmd_str)
    ret = os.system(cmd_str)  # rev.2: os.system → line-buffered stream
    elapsed = (time.time() - t0) / 60
    if ret != 0:
        print(f'\n[FAIL] {r["cell_id"]} seed={r["seed"]} exit={ret} ({elapsed:.1f} min)')
        break
    print(f'[done] {r["cell_id"]} seed={r["seed"]} → {ckpt_dir} ({elapsed:.1f} min)')

## 3. Evaluation (4 runs × 100 episodes against fixed manifest)

训练时关掉了 `--eval-every`，这里 separately 跑 `scripts.evaluate_offline` 对每个 checkpoint 走完整 100 ep manifest。

**rev.3 改路径**：`--output-json` 写到 `results/offline/rebrac/broad_validation_v2/<cell>/seed_<n>/test_result.json`（不在 checkpoint dir）。Skip-resume：检查新位置存在性。

In [ ]:
for i, r in enumerate(RUNS, 1):
    ckpt_dir = Path(r['ckpt_dir'])
    result_dir = Path(r['result_dir'])
    test_json = result_dir / 'test_result.json'

    print(f'\n========== [{i}/{len(RUNS)}] eval {r["cell_id"]} seed={r["seed"]} ==========')

    if test_json.exists():
        print(f'[skip] test_result.json exists: {test_json}')
        continue
    if not (ckpt_dir / 'agent_final.pt').exists():
        print(f'[skip] no agent_final.pt at {ckpt_dir} (training not done?)')
        continue

    result_dir.mkdir(parents=True, exist_ok=True)

    cmd_parts = [
        'python', '-u', '-m', 'scripts.evaluate_offline',
        '--checkpoint', str(ckpt_dir),
        '--agent-file', 'agent_final.pt',
        '--manifest', r['manifest'],
        '--episodes', '100',
        '--seed', '123',
        '--device', 'cuda',
        '--num-workers', '4',
        '--worker-device', 'cpu',
        '--output-json', str(test_json),
    ]
    cmd_str = ' '.join(shlex.quote(p) for p in cmd_parts)
    print(cmd_str)
    ret = os.system(cmd_str)
    if ret != 0:
        print(f'[FAIL] {r["cell_id"]} seed={r["seed"]} eval exit={ret}')
        break
    print(f'[done] {test_json}')

## 4. Raw JSON dump（诊断 — 4 个 test_result.json 字段全表）

**rev.2 新增**：直接打印每个 test_result.json 的所有标量字段，避免 §5 summary 出 bug 时不知道哪里坏了。

In [ ]:
import json
for r in RUNS:
    test_json = Path(r['result_dir']) / 'test_result.json'
    print(f'\n=== {r["cell_id"]} seed={r["seed"]} → {test_json} ===')
    if not test_json.exists():
        print('  MISSING')
        continue
    d = json.loads(test_json.read_text())
    for k in sorted(d.keys()):
        v = d[k]
        if isinstance(v, (int, float, str, bool, type(None))):
            print(f'  {k}: {v}')
        elif isinstance(v, list):
            print(f'  {k}: list(len={len(v)})')
        elif isinstance(v, dict):
            print(f'  {k}: dict(keys={list(v.keys())[:5]}{"..." if len(v) > 5 else ""})')
        else:
            print(f'  {k}: <{type(v).__name__}>')

## 5. Systematic results + verdict gate

**rev.2 fix**：
- `succ = d.get('eval_success_rate')` 改成显式 None check + NaN 防护（rev.1 用 `or` 短路，0.0 被当 falsy 丢值）
- 表格补 v1-style 7 列：`n / mean / std / Δ_vs_eff_v2_anchor / per_seed / avg_return / avg_safety_cost`

**rev.3 改输出位置**：写 `results/offline/rebrac/broad_validation_v2/summaries/{verdict_decision.json, p1_overview.csv}` 留痕（v1 同款）

In [ ]:
import csv
import json
import statistics

ANCHOR_EFF_V2_MEAN = 0.902  # main-line efficiency_v2 5-seed anchor
ANCHOR_EFF_V2_STD = 0.021
ONLINE_CATASTROPHIC = 0.10  # online §7.6 single_cross_s0 floor (cross_u15/s0)
ORACLE_CEILING = 0.70  # privileged S sanity ceiling (cross_u15/s0/privileged/30 ep)

def _safe_float(x):
    if x is None:
        return None
    try:
        f = float(x)
        return None if f != f else f  # NaN check
    except (TypeError, ValueError):
        return None

rows_by_cell = {}
for r in RUNS:
    test_json = Path(r['result_dir']) / 'test_result.json'
    if not test_json.exists():
        print(f'[missing] {test_json}')
        continue
    d = json.loads(test_json.read_text())
    succ = _safe_float(d.get('eval_success_rate'))
    if succ is None:
        succ = _safe_float(d.get('success_rate'))
    if succ is None:
        print(f'[warn] {test_json} has no usable success rate; keys={list(d.keys())}')
        continue
    rows_by_cell.setdefault(r['cell_id'], []).append({
        'seed': r['seed'],
        'success': succ,
        'avg_return': _safe_float(d.get('eval_avg_return')),
        'avg_safety_cost': _safe_float(d.get('eval_avg_safety_cost')),
        'avg_time_s': _safe_float(d.get('eval_avg_time_s')),
        'avg_path_efficiency': _safe_float(d.get('eval_avg_path_efficiency')),
    })

print('=' * 120)
header = f'{"cell":<6}{"n":>4}{"succ_mean":>11}{"succ_std":>10}{"Δ_vs_eff_v2":>14}{"per_seed":>24}{"avg_return":>13}{"avg_safety":>12}'
print(header)
print('-' * 120)

agg = {}
for cell_id, results in rows_by_cell.items():
    succ_list = [x['success'] for x in results]
    if not succ_list:
        continue
    mean = statistics.mean(succ_list)
    std = statistics.stdev(succ_list) if len(succ_list) > 1 else 0.0
    delta = mean - ANCHOR_EFF_V2_MEAN
    per_seed_pairs = [(x['seed'], x['success']) for x in results]
    per_seed_str = ' '.join(f'{s}={v:.3f}' for s, v in per_seed_pairs)
    ret_list = [x['avg_return'] for x in results if x['avg_return'] is not None]
    safety_list = [x['avg_safety_cost'] for x in results if x['avg_safety_cost'] is not None]
    avg_R = statistics.mean(ret_list) if ret_list else float('nan')
    avg_safety = statistics.mean(safety_list) if safety_list else float('nan')
    agg[cell_id] = {
        'n_seeds': len(succ_list),
        'success_mean': mean,
        'success_std': std,
        'delta_vs_eff_v2_anchor': delta,
        'per_seed': per_seed_pairs,
        'avg_return': avg_R,
        'avg_safety_cost': avg_safety,
    }
    print(f'{cell_id:<6}{len(succ_list):>4}{mean:>11.4f}{std:>10.4f}{delta*100:>+12.2f}pp{per_seed_str:>24}{avg_R:>13.2f}{avg_safety:>12.3f}')
print('=' * 120)
print(f'Anchor (mainline efficiency_v2 5-seed): mean={ANCHOR_EFF_V2_MEAN:.3f}, std={ANCHOR_EFF_V2_STD:.3f}')
print(f'Online catastrophic floor (cross_u15/s0): {ONLINE_CATASTROPHIC:.2f}')
print(f'Oracle ceiling (privileged sanity, cross_u15/s0/30 ep): {ORACLE_CEILING:.2f}')

# ----- §5.2 N0 verdict -----
verdict = {}
print('\n--- §5.2 N0 verdict (reward bridge: arrival_v2 vs efficiency_v2 anchor 0.902) ---')
if 'N0' in agg:
    n0_mean = agg['N0']['success_mean']
    n0_delta_pp = (n0_mean - ANCHOR_EFF_V2_MEAN) * 100
    if n0_mean >= 0.70:
        n0_v = 'HOLDS'
        n0_note = 'reward bridge holds — proceed to N2\' analysis'
    elif n0_mean >= 0.50:
        n0_v = 'WEAK'
        n0_note = f'weak bridge — N2\' 仍跑，paper 写作时标明 arrival_v2 退化 {n0_delta_pp:+.2f}pp'
    else:
        n0_v = 'FAIL'
        n0_note = 'reward bridge FAILS — 暂停所有后续 cell，review reward/collector/dataset'
    verdict['N0'] = {'mean': n0_mean, 'delta_pp': n0_delta_pp, 'verdict': n0_v, 'note': n0_note}
    print(f'  N0 mean = {n0_mean:.4f}  Δ = {n0_delta_pp:+.2f}pp  → {n0_v}')
    print(f'  {n0_note}')

# ----- §5.3 N2' verdict -----
print('\n--- §5.3 N2\' verdict (oracle-teacher partial-obs ceiling) ---')
if 'N2p' in agg:
    n2p_mean = agg['N2p']['success_mean']
    lift_pp = (n2p_mean - ONLINE_CATASTROPHIC) * 100
    recovery_pct = n2p_mean / ORACLE_CEILING * 100
    if n2p_mean >= 0.40:
        n2p_v = 'STRONG_POSITIVE'
        n2p_note = 'offline RL with oracle demos meaningfully bridges partial-obs gap; s0-conditioned imitation extracts ≥ half of oracle ceiling'
    elif n2p_mean >= 0.15:
        n2p_v = 'PARTIAL'
        n2p_note = f'recovers {recovery_pct:.1f}% of oracle teacher → trigger M1 BC sweep (β1 ∈ {{0,1,2,4,8}} × 3 seed = 15 runs ≈ 15h L4)'
    else:
        n2p_v = 'STRONG_NEGATIVE'
        n2p_note = 'critical regime is actor-fundamental under s0; even oracle demos cannot bridge partial-obs gap when actor lacks hull-integral flow'
    verdict['N2p'] = {
        'mean': n2p_mean,
        'lift_vs_online_floor_pp': lift_pp,
        'recovery_of_oracle_pct': recovery_pct,
        'verdict': n2p_v,
        'note': n2p_note,
    }
    print(f'  N2\' mean = {n2p_mean:.4f}  vs online floor {ONLINE_CATASTROPHIC:.2f} → lift = {lift_pp:+.2f}pp  vs oracle {ORACLE_CEILING:.2f} → recovery = {recovery_pct:.1f}%')
    print(f'  → {n2p_v}')
    print(f'  {n2p_note}')

# ----- M1 trigger decision -----
print('\n--- M1 BC sweep trigger ---')
if verdict.get('N2p', {}).get('verdict') == 'PARTIAL':
    print('  → TRIGGER M1: 另起 notebook 跑 β1 ∈ {0,1,2,4,8} × 3 seed = 15 runs')
    verdict['M1_trigger'] = True
else:
    print('  → NO TRIGGER (N2\' not in [0.15, 0.40] partial zone)')
    verdict['M1_trigger'] = False

print('\n--- 2-seed limitation reminder ---')
print('  本轮只跑 2 seed [42, 0]，std df=1 极不稳。若 N2\' 落入 partial zone 或论文审稿要 3-seed power，')
print('  补 seed 43 后重跑 §2 + §3 + §5（skip-resume 自动跳过 42 / 0）。')

# ----- 写 verdict_decision.json + p1_overview.csv 留痕 -----
verdict_path = SUMMARIES_DIR / 'verdict_decision.json'
verdict_path.write_text(json.dumps({
    'anchor': {'efficiency_v2_mean': ANCHOR_EFF_V2_MEAN, 'efficiency_v2_std': ANCHOR_EFF_V2_STD,
               'online_catastrophic_floor': ONLINE_CATASTROPHIC, 'oracle_ceiling': ORACLE_CEILING},
    'seeds': SEEDS,
    'cells': agg,
    'verdict': verdict,
}, indent=2, default=lambda o: list(o) if isinstance(o, tuple) else str(o)))
print(f'\n[wrote] {verdict_path}')

csv_path = SUMMARIES_DIR / 'p1_overview.csv'
with csv_path.open('w', newline='') as fp:
    w = csv.writer(fp)
    w.writerow(['cell_id', 'n_seeds', 'success_mean', 'success_std', 'delta_vs_eff_v2_pp',
                'per_seed_42', 'per_seed_0', 'avg_return', 'avg_safety_cost'])
    for cell_id, a in agg.items():
        per_seed_dict = dict(a['per_seed'])
        w.writerow([
            cell_id, a['n_seeds'],
            f'{a["success_mean"]:.4f}', f'{a["success_std"]:.4f}',
            f'{a["delta_vs_eff_v2_anchor"]*100:+.2f}',
            f'{per_seed_dict.get(42, float("nan")):.4f}',
            f'{per_seed_dict.get(0, float("nan")):.4f}',
            f'{a["avg_return"]:.2f}', f'{a["avg_safety_cost"]:.3f}',
        ])
print(f'[wrote] {csv_path}')

## 6. 跑完后清单（回到 local）

### Sync 回 git 仓库（只取 `results/`）

```bash
# 在本地 repo root：
rsync -av '<drive>/results/offline/rebrac/broad_validation_v2/' \
  results/offline/rebrac/broad_validation_v2/
```

包含：
- `results/offline/rebrac/broad_validation_v2/{N0,N2p}/seed_{42,0}/test_result.json` × 4
- `results/offline/rebrac/broad_validation_v2/summaries/{verdict_decision.json, p1_overview.csv}`

**checkpoint 不动**（agent_final.pt 大文件，gitignore；只在需要重 eval 或扩 seed 时回到 Drive）。

### 后续步骤

1. **更新** [`docs/rebrac_broad_validation_v2_plan.md`](../docs/rebrac_broad_validation_v2_plan.md) §10 status：填入实测 2-seed 数字 + verdict 决定
2. **新建** `docs/rebrac_broad_validation_v2_report.md`：N0 / N2' 数字 + §5.2 / §5.3 verdict 应用结果 + 是否触发 M1
3. **若 N2' partial**：另起 notebook 跑 M1 BC sweep（15 runs）
4. **若 N2' strong negative**：直接落 actor-fundamental ceiling 论断，broad val v2 收口
5. **若 reviewer push back 3-seed power**：本 notebook 加 seed 43 → 重跑 §2 + §3 + §5（skip-resume 自动跳过 42 / 0）